# A股成交量分析实验

本实验分析A股市场中成交量前5%的股票，并计算这些股票的成交量占当日总成交量的比例。

## 实验目标
- 识别每日成交量前5%的A股股票
- 计算这些股票的成交量相对于当日总成交量的比例
- 展示分析结果和趋势

In [54]:
# 简化版本 - 兼容性更好的初始化方式
import qlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from qlib.data import D
from datetime import datetime, timedelta
import math
import time

# 简单初始化Qlib（兼容性最好）
qlib.init(provider_uri="~/.qlib/qlib_data/cn_data")
print("Qlib初始化完成（使用本地数据源）")
print("注意：缓存功能需要根据Qlib版本进行配置")


[29264:MainThread](2025-09-23 13:58:58,935) INFO - qlib.Initialization - [config.py:452] - default_conf: client.
[29264:MainThread](2025-09-23 13:58:58,936) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.
[29264:MainThread](2025-09-23 13:58:58,937) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': WindowsPath('C:/Users/zihao/.qlib/qlib_data/cn_data')}


Qlib初始化完成（使用本地数据源）
注意：缓存功能需要根据Qlib版本进行配置


In [ ]:
# 简化实验参数设置
print("=== 实验参数设置 ===")

# 基本参数
MARKET = "csi300"  # 使用CSI300股票池
START_DATE = "2023-01-01"
END_DATE = "2023-12-31"
TOP_VOLUME_RATIO = 0.05  # 前5%

print(f"分析期间: {START_DATE} 到 {END_DATE}")
print(f"分析股票池: {MARKET}")
print(f"成交量排名前 {TOP_VOLUME_RATIO*100}% 的股票")
print("参数设置完成！")


=== 性能优化配置 ===
时间范围优化：使用最近6个月数据 (2025-03-27 到 2025-09-23)
股票池优化：使用 csi100 股票池（数据量更小）
批量大小：50 只股票/批次
优化配置完成！


In [56]:
# 实验参数设置
MARKET = "csi300"  # 使用CSI300股票池
START_DATE = "2025-01-01"
END_DATE = "2025-12-31"
TOP_VOLUME_RATIO = 0.05  # 前5%


print(f"分析期间: {START_DATE} 到 {END_DATE}")
print(f"分析股票池: {MARKET}")
print(f"成交量排名前 {TOP_VOLUME_RATIO*100}% 的股票")

分析期间: 2025-01-01 到 2025-12-31
分析股票池: csi300
成交量排名前 5.0% 的股票


In [58]:
# 简化数据获取 - 直接获取所有数据
print("正在获取A股所有股票成交量数据...")
print("使用简化策略：直接获取 + 最小化字段")

# 获取股票池中的所有股票
print("正在获取股票池...")
start_time = time.time()
instruments = D.instruments(market=MARKET)
print(f"股票池获取完成，包含 {len(instruments)} 只股票，耗时: {time.time() - start_time:.2f}秒")

# 最小化字段 - 只获取成交量数据
fields = ['$volume']  # 只获取成交量，大幅减少I/O
print(f"字段最小化：只获取 {fields}")
print(f"分析期间: {START_DATE} 到 {END_DATE}")

# 直接获取所有数据
print("开始获取数据...")
data_start_time = time.time()

try:
    raw_data = D.features(
        instruments=instruments,
        fields=fields,
        start_time=START_DATE,
        end_time=END_DATE
    )
    print(f"数据获取完成，耗时: {time.time() - data_start_time:.2f}秒")
except Exception as e:
    print(f"数据获取失败: {e}")
    raw_data = pd.DataFrame()

print(f"\n最终数据形状: {raw_data.shape}")
if not raw_data.empty:
    print(f"数据时间范围: {raw_data.index.get_level_values('datetime').min()} 到 {raw_data.index.get_level_values('datetime').max()}")
    print(f"股票数量: {len(raw_data.index.get_level_values('instrument').unique())}")
    print("\n数据样本:")
    print(raw_data.head(10))
else:
    print("数据为空，请检查数据源和参数设置")

正在获取A股所有股票成交量数据...
使用简化策略：直接获取 + 最小化字段
正在获取股票池...
股票池获取完成，包含 2 只股票，耗时: 0.00秒
字段最小化：只获取 ['$volume']
分析期间: 2025-01-01 到 2025-12-31
开始获取数据...
数据获取完成，耗时: 25.75秒

最终数据形状: (49978, 1)
数据时间范围: 2025-01-02 00:00:00 到 2025-09-08 00:00:00
股票数量: 308

数据样本:
                             $volume
instrument datetime                 
SH600000   2025-01-02  525178.312500
           2025-01-03  337406.468750
           2025-01-06  531136.375000
           2025-01-07  267998.750000
           2025-01-08  321048.000000
           2025-01-09  261057.140625
           2025-01-10  212538.453125
           2025-01-13  254373.687500
           2025-01-14  257303.937500
           2025-01-15  255979.468750


In [59]:
# 诊断数据问题
print("=== 数据诊断 ===")
if raw_data.empty:
    print("数据为空，无法进行诊断")
else:
    print(f"原始数据形状: {raw_data.shape}")
    print(f"原始数据索引: {raw_data.index.names}")
    print(f"原始数据列名: {raw_data.columns.tolist()}")

    # 检查索引内容
    print(f"\n索引内容示例:")
    print(f"instrument索引示例: {raw_data.index.get_level_values('instrument')[:5].tolist()}")
    print(f"datetime索引示例: {raw_data.index.get_level_values('datetime')[:5].tolist()}")

    # 重置索引并检查
    volume_df = raw_data.reset_index()
    print(f"\n重置索引后形状: {volume_df.shape}")
    print(f"重置索引后列名: {volume_df.columns.tolist()}")
    print(f"重置索引后前5行:")
    print(volume_df.head())

    # 正确设置列名（现在只有volume字段）
    if len(volume_df.columns) == 3:  # instrument, datetime, volume
        volume_df.columns = ['instrument', 'datetime', 'volume']
    else:  # 如果还有其他字段，保持原有逻辑
        volume_df.columns = ['instrument', 'datetime'] + [f'field_{i}' for i in range(len(volume_df.columns)-2)]
        if len(volume_df.columns) >= 3:
            volume_df = volume_df.rename(columns={volume_df.columns[2]: 'volume'})
    
    print(f"\n设置列名后:")
    print(volume_df.head())
    print(f"数据类型:")
    print(volume_df.dtypes)


=== 数据诊断 ===
原始数据形状: (49978, 1)
原始数据索引: ['instrument', 'datetime']
原始数据列名: ['$volume']

索引内容示例:
instrument索引示例: ['SH600000', 'SH600000', 'SH600000', 'SH600000', 'SH600000']
datetime索引示例: [Timestamp('2025-01-02 00:00:00'), Timestamp('2025-01-03 00:00:00'), Timestamp('2025-01-06 00:00:00'), Timestamp('2025-01-07 00:00:00'), Timestamp('2025-01-08 00:00:00')]

重置索引后形状: (49978, 3)
重置索引后列名: ['instrument', 'datetime', '$volume']
重置索引后前5行:
  instrument   datetime       $volume
0   SH600000 2025-01-02  525178.31250
1   SH600000 2025-01-03  337406.46875
2   SH600000 2025-01-06  531136.37500
3   SH600000 2025-01-07  267998.75000
4   SH600000 2025-01-08  321048.00000

设置列名后:
  instrument   datetime        volume
0   SH600000 2025-01-02  525178.31250
1   SH600000 2025-01-03  337406.46875
2   SH600000 2025-01-06  531136.37500
3   SH600000 2025-01-07  267998.75000
4   SH600000 2025-01-08  321048.00000
数据类型:
instrument            object
datetime      datetime64[ns]
volume               float32
dtype: 

In [60]:
# 简化数据统计
print("=== 数据统计 ===")

if not raw_data.empty:
    total_records = len(raw_data)
    total_stocks = len(raw_data.index.get_level_values('instrument').unique())
    total_days = len(raw_data.index.get_level_values('datetime').unique())
    
    print(f"数据统计:")
    print(f"  总记录数: {total_records:,}")
    print(f"  股票数量: {total_stocks}")
    print(f"  交易日数: {total_days}")
    print(f"  平均每只股票记录数: {total_records/total_stocks:.1f}")
else:
    print("数据为空，无法进行统计")


=== 数据统计 ===
数据统计:
  总记录数: 49,978
  股票数量: 308
  交易日数: 167
  平均每只股票记录数: 162.3


In [62]:
# 计算每日成交量前5%股票的分析
print("正在分析每日成交量前5%股票...")

# 检查数据结构
print(f"原始数据形状: {raw_data.shape}")
print(f"原始数据列名: {raw_data.columns.tolist()}")
print(f"原始数据索引: {raw_data.index.names}")

# 重置索引以便操作
volume_df = raw_data.reset_index()

# 根据实际列数设置列名
# 注意：从输出看，索引顺序是 ['instrument', 'datetime']，所以重置后是 ['instrument', 'datetime', ...]
if len(volume_df.columns) == 8:  # instrument, datetime + 6个字段
    volume_df.columns = ['instrument', 'datetime', 'open', 'high', 'low', 'close', 'volume', 'factor']
elif len(volume_df.columns) == 7:  # instrument, datetime + 5个字段
    volume_df.columns = ['instrument', 'datetime', 'open', 'high', 'low', 'close', 'volume']
else:
    # 如果列数不匹配，先显示实际列名
    print(f"实际列数: {len(volume_df.columns)}")
    print(f"实际列名: {volume_df.columns.tolist()}")
    # 手动设置列名
    volume_df.columns = ['instrument', 'datetime'] + [f'field_{i}' for i in range(len(volume_df.columns)-2)]
    # 假设volume是第6列（索引5）
    if len(volume_df.columns) >= 6:
        volume_df = volume_df.rename(columns={volume_df.columns[5]: 'volume'})

print(f"重置索引后数据形状: {volume_df.shape}")
print(f"重置索引后列名: {volume_df.columns.tolist()}")

# 检查数据类型和样本数据
print(f"\n数据类型:")
print(volume_df.dtypes)
print(f"\n前5行数据:")
print(volume_df.head())

# 确保datetime列是datetime类型
if 'datetime' in volume_df.columns:
    # 如果datetime列不是datetime类型，尝试转换
    if not pd.api.types.is_datetime64_any_dtype(volume_df['datetime']):
        print("正在转换datetime列...")
        volume_df['datetime'] = pd.to_datetime(volume_df['datetime'], errors='coerce')
    
    # 去除转换失败的记录
    volume_df = volume_df.dropna(subset=['datetime'])

# 去除空值
volume_df = volume_df.dropna()

print(f"去除空值后数据形状: {volume_df.shape}")

# 按日期分组计算每日指标
daily_analysis = []

for date in volume_df['datetime'].unique():
    daily_data = volume_df[volume_df['datetime'] == date].copy()
    
    if len(daily_data) == 0:
        continue
    
    # 计算当日总成交量
    total_volume = daily_data['volume'].sum()
    
    # 计算前5%股票数量
    top_count = max(1, int(len(daily_data) * TOP_VOLUME_RATIO))
    
    # 按成交量排序，获取前5%
    top_stocks = daily_data.nlargest(top_count, 'volume')
    
    # 计算前5%股票的总成交量
    top_volume = top_stocks['volume'].sum()
    
    # 计算比例
    volume_ratio = top_volume / total_volume if total_volume > 0 else 0
    
    daily_analysis.append({
        'date': date,
        'total_volume': total_volume,
        'top_stocks_count': top_count,
        'top_volume': top_volume,
        'volume_ratio': volume_ratio,
        'total_stocks': len(daily_data)
    })

# 转换为DataFrame
analysis_df = pd.DataFrame(daily_analysis)

# 确保date列是datetime类型
if len(analysis_df) > 0:
    analysis_df['date'] = pd.to_datetime(analysis_df['date'])
    analysis_df = analysis_df.sort_values('date')

print(f"分析完成，共处理 {len(analysis_df)} 个交易日")
print("\n每日分析结果样本:")
analysis_df.head()

正在分析每日成交量前5%股票...
原始数据形状: (49978, 1)
原始数据列名: ['$volume']
原始数据索引: ['instrument', 'datetime']
实际列数: 3
实际列名: ['instrument', 'datetime', '$volume']
重置索引后数据形状: (49978, 3)
重置索引后列名: ['instrument', 'datetime', 'field_0']

数据类型:
instrument            object
datetime      datetime64[ns]
field_0              float32
dtype: object

前5行数据:
  instrument   datetime       field_0
0   SH600000 2025-01-02  525178.31250
1   SH600000 2025-01-03  337406.46875
2   SH600000 2025-01-06  531136.37500
3   SH600000 2025-01-07  267998.75000
4   SH600000 2025-01-08  321048.00000
去除空值后数据形状: (49908, 3)


KeyError: 'volume'

In [37]:
# 统计摘要
print("=== 成交量前5%股票分析统计摘要 ===")

# 检查是否有数据
if len(analysis_df) == 0:
    print("警告：没有有效的数据进行分析！")
    print("可能的原因：")
    print("1. 数据时间范围没有交易日")
    print("2. 数据转换过程中出现错误")
    print("3. 所有数据都被过滤掉了")
else:
    print(f"分析期间: {analysis_df['date'].min().strftime('%Y-%m-%d')} 到 {analysis_df['date'].max().strftime('%Y-%m-%d')}")
    print(f"交易日数量: {len(analysis_df)}")
    print(f"平均每日股票数量: {analysis_df['total_stocks'].mean():.1f}")
    print(f"平均前5%股票数量: {analysis_df['top_stocks_count'].mean():.1f}")
    print()
    print("=== 成交量比例统计 ===")
    print(f"前5%股票成交量占比 - 平均值: {analysis_df['volume_ratio'].mean():.4f} ({analysis_df['volume_ratio'].mean()*100:.2f}%)")
    print(f"前5%股票成交量占比 - 中位数: {analysis_df['volume_ratio'].median():.4f} ({analysis_df['volume_ratio'].median()*100:.2f}%)")
    print(f"前5%股票成交量占比 - 最小值: {analysis_df['volume_ratio'].min():.4f} ({analysis_df['volume_ratio'].min()*100:.2f}%)")
    print(f"前5%股票成交量占比 - 最大值: {analysis_df['volume_ratio'].max():.4f} ({analysis_df['volume_ratio'].max()*100:.2f}%)")
    print(f"前5%股票成交量占比 - 标准差: {analysis_df['volume_ratio'].std():.4f}")
    print()
    print("=== 总成交量统计 ===")
    print(f"日均总成交量: {analysis_df['total_volume'].mean():.0f}")
    print(f"总成交量中位数: {analysis_df['total_volume'].median():.0f}")

=== 成交量前5%股票分析统计摘要 ===
分析期间: 2025-01-02 到 2025-09-08
交易日数量: 167
平均每日股票数量: 298.9
平均前5%股票数量: 14.2

=== 成交量比例统计 ===
前5%股票成交量占比 - 平均值: 0.3282 (32.82%)
前5%股票成交量占比 - 中位数: 0.3236 (32.36%)
前5%股票成交量占比 - 最小值: 0.2520 (25.20%)
前5%股票成交量占比 - 最大值: 0.4288 (42.88%)
前5%股票成交量占比 - 标准差: 0.0335

=== 总成交量统计 ===
日均总成交量: 897276544
总成交量中位数: 819373888


In [38]:
# 创建可视化图表
if len(analysis_df) == 0:
    print("无法创建图表：没有有效的数据")
    print("请检查数据获取和处理过程")
else:
    fig = make_subplots(
        rows=3, cols=1,
        subplot_titles=(
            'A股前5%成交量股票的成交量占比时间序列',
            '每日总成交量',
            '成交量占比分布直方图'
        ),
        vertical_spacing=0.08
    )

    # 第1个子图: 成交量占比时间序列
    fig.add_trace(
        go.Scatter(
            x=analysis_df['date'],
            y=analysis_df['volume_ratio'] * 100,
            mode='lines+markers',
            name='成交量占比 (%)',
            line=dict(color='blue', width=2),
            marker=dict(size=4)
        ),
        row=1, col=1
    )

    # 添加平均线
    mean_ratio = analysis_df['volume_ratio'].mean() * 100
    fig.add_hline(
        y=mean_ratio,
        line_dash="dash",
        line_color="red",
        annotation_text=f"平均值: {mean_ratio:.2f}%",
        row=1, col=1
    )

    # 第2个子图: 总成交量时间序列
    fig.add_trace(
        go.Scatter(
            x=analysis_df['date'],
            y=analysis_df['total_volume'],
            mode='lines',
            name='总成交量',
            line=dict(color='green', width=2)
        ),
        row=2, col=1
    )

    # 第3个子图: 分布直方图
    fig.add_trace(
        go.Histogram(
            x=analysis_df['volume_ratio'] * 100,
            nbinsx=30,
            name='占比分布',
            marker_color='orange'
        ),
        row=3, col=1
    )

    # 更新布局
    fig.update_layout(
        height=900,
        title_text="A股成交量前5%股票分析结果",
        title_x=0.5,
        showlegend=False
    )

    # 更新x轴和y轴标签
    fig.update_xaxes(title_text="日期", row=1, col=1)
    fig.update_yaxes(title_text="成交量占比 (%)", row=1, col=1)

    fig.update_xaxes(title_text="日期", row=2, col=1)
    fig.update_yaxes(title_text="总成交量", row=2, col=1)

    fig.update_xaxes(title_text="成交量占比 (%)", row=3, col=1)
    fig.update_yaxes(title_text="频次", row=3, col=1)

    fig.show()

In [39]:
# 月度分析
print("=== 月度分析 ===")

if len(analysis_df) == 0:
    print("无法进行月度分析：没有有效的数据")
else:
    analysis_df['year_month'] = analysis_df['date'].dt.to_period('M')
    monthly_stats = analysis_df.groupby('year_month').agg({
        'volume_ratio': ['mean', 'std', 'min', 'max'],
        'total_volume': 'mean',
        'date': 'count'
    }).round(4)

    monthly_stats.columns = ['平均占比', '占比标准差', '最小占比', '最大占比', '平均总成交量', '交易日数']
    monthly_stats['平均占比(%)'] = monthly_stats['平均占比'] * 100
    monthly_stats['占比标准差(%)'] = monthly_stats['占比标准差'] * 100

    print(monthly_stats[['平均占比(%)', '占比标准差(%)', '交易日数']])

=== 月度分析 ===
              平均占比(%)  占比标准差(%)  交易日数
year_month                           
2025-01     37.310001      3.42    18
2025-02     35.989998      2.28    18
2025-03     31.709999      2.07    21
2025-04     31.020000      2.52    21
2025-05     29.070000      2.22    19
2025-06     32.520000      2.23    20
2025-07     32.709999      2.42    23
2025-08     33.489998      2.32    21
2025-09     31.090000      1.71     6


In [40]:
# 特殊日期分析 - 找出成交量占比异常的日期
print("=== 异常日期分析 ===")

if len(analysis_df) == 0:
    print("无法进行异常日期分析：没有有效的数据")
else:
    # 计算分位数
    q95 = analysis_df['volume_ratio'].quantile(0.95)
    q05 = analysis_df['volume_ratio'].quantile(0.05)

    print(f"成交量占比95%分位数: {q95*100:.2f}%")
    print(f"成交量占比5%分位数: {q05*100:.2f}%")

    # 找出异常高的日期
    high_ratio_days = analysis_df[analysis_df['volume_ratio'] > q95].copy()
    high_ratio_days = high_ratio_days.sort_values('volume_ratio', ascending=False)

    print("\n成交量占比最高的10个交易日:")
    display_cols = ['date', 'volume_ratio', 'total_volume', 'top_stocks_count']
    high_ratio_display = high_ratio_days[display_cols].head(10).copy()
    high_ratio_display['volume_ratio'] = high_ratio_display['volume_ratio'] * 100
    high_ratio_display.columns = ['日期', '成交量占比(%)', '总成交量', '前5%股票数量']
    print(high_ratio_display.to_string(index=False))

    # 找出异常低的日期
    low_ratio_days = analysis_df[analysis_df['volume_ratio'] < q05].copy()
    low_ratio_days = low_ratio_days.sort_values('volume_ratio')

    print("\n成交量占比最低的10个交易日:")
    low_ratio_display = low_ratio_days[display_cols].head(10).copy()
    low_ratio_display['volume_ratio'] = low_ratio_display['volume_ratio'] * 100
    low_ratio_display.columns = ['日期', '成交量占比(%)', '总成交量', '前5%股票数量']
    print(low_ratio_display.to_string(index=False))

=== 异常日期分析 ===
成交量占比95%分位数: 38.43%
成交量占比5%分位数: 27.94%

成交量占比最高的10个交易日:
        日期  成交量占比(%)        总成交量  前5%股票数量
2025-01-17 42.882149 736477184.0       15
2025-01-22 42.310722 759314688.0       15
2025-01-16 42.052197 859737856.0       14
2025-02-05 41.157921 911178880.0       15
2025-01-21 39.899506 718549952.0       15
2025-01-23 39.237972 954632640.0       15
2025-01-09 38.857590 706203072.0       15
2025-01-27 38.511532 778612096.0       15
2025-02-06 38.452106 919899136.0       14

成交量占比最低的10个交易日:
        日期  成交量占比(%)        总成交量  前5%股票数量
2025-04-30 25.199717 686870080.0       14
2025-05-20 25.593117 518877952.0       14
2025-05-23 25.805309 594118016.0       14
2025-05-06 26.309446 754587072.0       14
2025-05-26 26.659077 525253504.0       14
2025-04-29 26.680643 551544320.0       14
2025-05-22 27.153980 523277312.0       14
2025-03-19 27.724716 691870720.0       14
2025-05-16 27.857382 594489280.0       14


In [41]:
# 保存分析结果
if len(analysis_df) == 0:
    print("无法保存结果：没有有效的数据")
    print("请检查数据获取和处理过程")
else:
    output_file = '../experiments/volume_analysis_results.csv'
    analysis_df.to_csv(output_file, index=False, encoding='utf-8-sig')
    print(f"\n分析结果已保存到: {output_file}")

    # 显示核心指标摘要
    print("\n=== 核心指标摘要 ===")
    print(f"分析期间: {START_DATE} 到 {END_DATE}")
    print(f"股票池: {MARKET}")
    print(f"前5%股票平均成交量占比: {analysis_df['volume_ratio'].mean()*100:.2f}%")
    print(f"该指标标准差: {analysis_df['volume_ratio'].std()*100:.2f}%")
    print(f"该指标变异系数: {analysis_df['volume_ratio'].std()/analysis_df['volume_ratio'].mean():.3f}")

print("\n实验完成！")


分析结果已保存到: ../experiments/volume_analysis_results.csv

=== 核心指标摘要 ===
分析期间: 2025-01-01 到 2025-12-31
股票池: csi300
前5%股票平均成交量占比: 32.82%
该指标标准差: 3.35%
该指标变异系数: 0.102

实验完成！
